# VRP

## Greedy

In [14]:
import math


def read_instance(path):
    with open(path, "r", encoding="utf-8") as f:
        n, m, c = map(int, f.readline().split())

        demands = [0] * n
        coords = [None] * n

        for i in range(n):
            d, x, y = map(float, f.readline().split())
            demands[i] = int(d)
            coords[i] = (x, y)

    return n, m, c, demands, coords


def greedy_vrp(depot, customers, demands, capacity):
    unvisited = set(range(len(customers)))
    routes = []

    while unvisited:
        route, load, current = [], 0, depot

        while candidates := [
            i for i in unvisited
            if load + demands[i] <= capacity
        ]:
            i = min(candidates, key=lambda i: math.dist(current, customers[i]))
            route.append(i)
            load += demands[i]
            current = customers[i]
            unvisited.remove(i)

        routes.append(route)

    return routes


def total_distance(routes, depot, customers):
    total = 0.0

    for route in routes:
        if not route:
            continue

        total += math.dist(depot, customers[route[0]])

        for i in range(len(route) - 1):
            total += math.dist(customers[route[i]], customers[route[i + 1]])

        total += math.dist(customers[route[-1]], depot)

    return total


tests = [
    "data/vrp_16_3_1",
    "data/vrp_26_8_1",
    "data/vrp_51_5_1",
    "data/vrp_101_10_1",
    "data/vrp_200_16_1",
    "data/vrp_421_41_1",
]

for test in tests:
    n, m, c, demands, coords = read_instance(test)
    depot = coords[0]
    customers = coords[1:]
    customer_demands = demands[1:]
    routes = greedy_vrp(depot, customers, customer_demands, c)
    print(test, total_distance(routes, depot, customers))

data/vrp_16_3_1 316.7408960600777
data/vrp_26_8_1 719.3189434633235
data/vrp_51_5_1 711.4987137910165
data/vrp_101_10_1 1311.4998703102585
data/vrp_200_16_1 2034.3965714246424
data/vrp_421_41_1 2234.2062240291443


## 2-opt

In [15]:
def route_distance(route, depot, customers):
    if not route:
        return 0.0
    d = math.dist(depot, customers[route[0]])
    for k in range(len(route) - 1):
        d += math.dist(customers[route[k]], customers[route[k + 1]])
    d += math.dist(customers[route[-1]], depot)
    return d


def two_opt_route(route, depot, customers):
    if len(route) < 3:
        return route

    best = route[:]
    best_d = route_distance(best, depot, customers)
    improved = True

    while improved:
        improved = False
        for i in range(len(best) - 1):
            for j in range(i + 1, len(best)):
                new = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
                new_d = route_distance(new, depot, customers)

                if new_d < best_d:
                    best, best_d = new, new_d
                    improved = True
                    break
            if improved:
                break
    return best

tests = [
    "data/vrp_16_3_1",
    "data/vrp_26_8_1",
    "data/vrp_51_5_1",
    "data/vrp_101_10_1",
    "data/vrp_200_16_1",
    "data/vrp_421_41_1",
]

for test in tests:
    n, m, c, demands, coords = read_instance(test)
    depot = coords[0]
    customers = coords[1:]
    customer_demands = demands[1:]
    routes = greedy_vrp(depot, customers, customer_demands, c)
    routes = [two_opt_route(r, depot, customers) for r in routes]
    total = sum(route_distance(r, depot, customers) for r in routes)

    print(test, total)

data/vrp_16_3_1 284.41850585052373
data/vrp_26_8_1 702.3121288788149
data/vrp_51_5_1 665.0385752078685
data/vrp_101_10_1 1250.416618467446
data/vrp_200_16_1 1953.5894732964368
data/vrp_421_41_1 2138.181735685854


vrp_101_10_1 не проходит ни на 3, ни на 5, остальные проходят на 3.